# 🌲 Random Forests - In Depth

Decision Trees are great, but they are unstable. A tiny change in the data can completely redraw the entire tree.

Random Forests solve this using a technique called **Ensemble Learning**—specifically **Bagging** (Bootstrap Aggregating). It builds hundreds of slightly different trees and asks them to vote.

## 🧠 1. Deep Dive into the Theory

If you ask 1 person to guess the weight of a cow, they might be off by 500 lbs. If you ask 1,000 people and take the average, they will be shockingly close to the exact weight. This is the **Law of Large Numbers**.

For this to work in ML, the trees must be independent and make *different* errors. If we train 100 trees on the exact same data, they will all make the exact same mistakes. Random Forests inject randomness in two ways:

1. **Bootstrapping (Row Randomness)**: Every tree gets a random sample of the training data (drawn *with replacement*).
2. **Feature Randomness**: When a tree is trying to split a node, it is not allowed to look at all features! It is only allowed to look at a random subset (usually $\sqrt{\text{total features}}$). This prevents a single dominant feature from controlling every single tree.

## 💻 2. Implementation & Out-Of-Bag (OOB) Evaluation

Because of Bootstrapping, each tree only sees about 63% of the training data. The remaining 37% is called **Out-Of-Bag (OOB)** data.
We can use this leftover data to test the trees instantly without needing a separate validation set!

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Using California Housing Dataset (Regression Example)
data = fetch_california_housing()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Train a Random Forest Regressor
# oob_score=True tells it to evaluate itself on the unseen 37% of data during training
rf = RandomForestRegressor(n_estimators=100, oob_score=True, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

print(f"Out-Of-Bag (OOB) Score (R^2): {rf.oob_score_:.4f}")

y_pred = rf.predict(X_test)
print(f"Actual Test Set Score (R^2): {r2_score(y_test, y_pred):.4f}")
print("Notice how incredibly accurate the OOB score is at predicting test performance without ever seeing the test set!")

## 🔍 3. The Power of `n_estimators`

Does adding more trees always make the model better? Let's find out.

In [ ]:
# Warning: This block might take a few seconds to run
estimators_list = [10, 50, 100, 200]
r2_scores = []

for n in estimators_list:
    temp_rf = RandomForestRegressor(n_estimators=n, random_state=42, n_jobs=-1)
    temp_rf.fit(X_train, y_train)
    r2_scores.append(r2_score(y_test, temp_rf.predict(X_test)))

plt.figure(figsize=(8, 5))
plt.plot(estimators_list, r2_scores, marker='o', color='forestgreen')
plt.title('Random Forest Performance vs. Number of Trees')
plt.xlabel('Number of Trees (n_estimators)')
plt.ylabel('R-squared Score')
plt.show()
print("Key Takeaway: Adding more trees improves performance, but it eventually plateaus. You can't overfit by adding more trees, but you do waste compute time!")

## 🧐 4. Extracting Business Insights (Feature Importance)

Random Forests excel at telling you exactly what drives your target variable.

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
importances.plot(kind='bar', color='darkorange', edgecolor='k')
plt.title('Feature Importances for California Housing Prices')
plt.ylabel('Relative Importance')
plt.xticks(rotation=45)
plt.show()
print("MedInc (Median Income) is by far the biggest driver of house prices in this dataset.")

## 📊 5. Summary: Pros and Cons

| Pros | Cons |
|------|------|
| Incredibly accurate and robust to outliers | "Black Box" model (you can't draw the 100 trees easily) |
| Automatically handles non-linear relationships | Very slow to train on massive datasets with thousands of trees |
| Provides Feature Importances out of the box | Can be massive in memory size |
| Requires almost no hyperparameter tuning to get a "good" result | Doesn't predict well beyond the range of training data (extrapolation) |